# 1. Setup: Install dependencies and import libraries

In [1]:
!pip install pandas_datareader

import pandas as pd
import yfinance as yf
import requests
import io
from pandas_datareader import data as web
import datetime
import os

# 2. GENERIC DATA DOWNLOAD & CLEANING FUNCTION
# - Downloads price data
# - Extracts Close prices
# - Cleans missing data
# - Returns clean DataFrame

In [2]:
def download_and_clean(tickers, start="2014-01-01", end="2025-12-31"):

    print(f"Downloading {len(tickers)} tickers...")

    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        threads=True
    )

    close_prices = data["Close"]

    # Remove completely empty columns
    df = close_prices.dropna(axis=1, how='all')

    # Keep assets with ≥95% data
    threshold = int(0.95 * len(df))
    df = df.dropna(axis=1, thresh=threshold)

    # Forward fill small gaps
    df = df.ffill()

    # Drop remaining NaNs
    df = df.dropna()

    print(f"Final shape: {df.shape}")

    return df

# 3.1 🇺🇸 S&P 500 — Ticker Extraction + Data Download

In [3]:
def get_sp500_tickers():
    """
    Scrape S&P 500 constituents from Wikipedia
    and format for Yahoo Finance
    """
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0'}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(response.text)
    df = tables[0]

    tickers = df['Symbol'].str.replace('.', '-', regex=False).tolist()
    return tickers


sp500_tickers = get_sp500_tickers()
sp500_clean = download_and_clean(sp500_tickers)

sp500_clean.to_csv("sp500_close_clean.csv")
print("Saved: sp500_close_clean.csv")

/tmp/ipykernel_9174/1187498659.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


[*********************100%***********************]  503 of 503 completed


Final shape: (2872, 460)
Saved: sp500_close_clean.csv


# 3.2 🇧🇷 Ibovespa (Brazil)

In [4]:
def get_bovespa_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_companies_listed_on_B3"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(response.text)

    df = tables[0]
    tickers = df["Ticker"].astype(str).str.strip() + ".SA"

    return tickers.tolist()


bovespa_tickers = get_bovespa_tickers()
bovespa_clean = download_and_clean(bovespa_tickers)

bovespa_clean.to_csv("bovespa_close_clean.csv")
print("Saved: bovespa_close_clean.csv")

/tmp/ipykernel_9174/3370599579.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
[                       0%                       ]ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AZUL4.SA"}}}
[****                   8%                       ]  7 of 88 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BRFS3.SA"}}}
[*********************100%***********************]  88 of 88 completed
ERROR:yfinance:
22 Failed downloads:
ERROR:yfinance:['AZUL4.SA', 'BRFS3.SA', 'NTCO3.SA', 'LCAM3.SA', 'CIEL3.SA', 'CCRO3.SA', 'BRML3.SA', 'VIIA3.SA', 'JBSS3.SA', 'ELET3.SA', 'BIDI11.SA', 'CRFB3.SA', 'SOMA3.SA', 'BPAN4.SA', 'CPLE6.SA', 'PETZ3.SA', 'MRFG3.SA', 'EMBR3.SA', 'EL

Final shape: (2970, 56)
Saved: bovespa_close_clean.csv


# 3.3 🇪🇺 EURO STOXX 50

In [5]:
def get_eurostoxx_tickers():
    url = "https://en.wikipedia.org/wiki/EURO_STOXX_50"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(io.StringIO(response.text))

    for t in tables:
        if "Ticker" in t.columns or "Symbol" in t.columns:
            col = "Ticker" if "Ticker" in t.columns else "Symbol"
            return t[col].astype(str).str.strip().tolist()


euro_tickers = get_eurostoxx_tickers()
euro_clean = download_and_clean(euro_tickers)

euro_clean.to_csv("eurostoxx50_close_clean.csv")
print("Saved: eurostoxx50_close_clean.csv")

[*********************100%***********************]  50 of 50 completed


Final shape: (2940, 46)
Saved: eurostoxx50_close_clean.csv


# 3.4 🇬🇧 FTSE 100

In [6]:
def get_ftse100_tickers():
    url = "https://en.wikipedia.org/wiki/FTSE_100_Index"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(io.StringIO(response.text))

    for t in tables:
        if "EPIC" in t.columns or "Ticker" in t.columns:
            col = "EPIC" if "EPIC" in t.columns else "Ticker"
            return (t[col].astype(str).str.strip() + ".L").tolist()


ftse_tickers = get_ftse100_tickers()
ftse_clean = download_and_clean(ftse_tickers)

ftse_clean.to_csv("ftse100_close_clean.csv")
print("Saved: ftse100_close_clean.csv")

[*********************100%***********************]  100 of 100 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BT.A.L']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (3031, 90)
Saved: ftse100_close_clean.csv


# 3.5 🇿🇦 JSE Top 40

In [7]:
jse_tickers = [
    "ABG.JO", "AGL.JO", "ANG.JO", "ANH.JO", "APN.JO", "BHG.JO", "BID.JO",
    "BVT.JO", "BTI.JO", "CPI.JO", "CLS.JO", "DSY.JO", "EXX.JO", "FSR.JO",
    "GLN.JO", "GFI.JO", "GRT.JO", "IMP.JO", "INL.JO", "INP.JO", "MNP.JO",
    "MRP.JO", "MTN.JO", "MCG.JO", "NPN.JO", "NED.JO", "NRP.JO", "NPH.JO",
    "OMU.JO", "PRX.JO", "RNI.JO", "REM.JO", "RMH.JO", "SLM.JO", "SOL.JO",
    "SHP.JO", "SBK.JO","SSW.JO", "VOD.JO", "WHL.JO"
]

jse_clean = download_and_clean(jse_tickers)

jse_clean.to_csv("jse40_close_clean.csv")
print("Saved: jse40_close_clean.csv")

[*********************100%***********************]  40 of 40 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MCG.JO']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (3037, 31)
Saved: jse40_close_clean.csv


# 🇯🇵 NIKKEI 225 — Japan

In [8]:
nikkei_tickers = [
    "6857.T","8267.T","5201.T","2802.T","6770.T","6113.T","9202.T",
    "8304.T","2502.T","3407.T","4503.T","7832.T","5108.T","7751.T",
    "6952.T","9022.T","9502.T","4519.T","7762.T","1721.T","7186.T",
    "8253.T","4751.T","7912.T","8750.T","4568.T","6367.T","1925.T",
    "8601.T","2432.T","4061.T","6902.T","4324.T","4631.T","5714.T",
    "9020.T","6361.T","4523.T","5020.T","6954.T","9983.T","6504.T",
    "4901.T","5803.T","6702.T","8354.T","5801.T","6674.T","1808.T",
    "7205.T","6305.T","7004.T","6501.T","7267.T","7741.T","5019.T",
    "7013.T","1605.T","3099.T","7202.T","8001.T","3086.T","9201.T",
    "8697.T","6178.T","2914.T","5411.T","1963.T","6473.T","1812.T",
    "4452.T","7012.T","9107.T","9433.T","9008.T","9009.T","6861.T",
    "2801.T","2503.T","5406.T","6301.T","9766.T","4902.T","6326.T",
    "3405.T","6971.T","4151.T","6920.T","4689.T","2413.T","8002.T",
    "8252.T","7261.T","2269.T","4385.T","6479.T","4188.T","8058.T",
    "6503.T","8802.T","7011.T","9301.T","5711.T","7211.T","8306.T",
    "8031.T","4183.T","8801.T","5706.T","9104.T","8411.T","8725.T",
    "6981.T","6701.T","3659.T","5333.T","2282.T","2871.T","6594.T",
    "7731.T","7974.T","5214.T","9147.T","3863.T","5401.T","9432.T",
    "9101.T","4021.T","7201.T","2002.T","1332.T","9843.T","6988.T",
    "8604.T","6471.T","6472.T","9613.T","1802.T","9007.T","3861.T",
    "6103.T","7733.T","6645.T","4661.T","8591.T","9532.T","4578.T",
    "5541.T","6752.T","4755.T","6098.T","6723.T","8308.T","4004.T",
    "7752.T","2501.T","7735.T","9735.T","6724.T","1928.T","3382.T",
    "6753.T","1803.T","4063.T","4507.T","4911.T","5831.T","6273.T",
    "9434.T","9984.T","2768.T","8630.T","6758.T","7270.T","3436.T",
    "4005.T","8053.T","5802.T","6302.T","5713.T","8316.T","8309.T",
    "5232.T","4506.T","8830.T","7269.T","8795.T","5233.T","1801.T",
    "6976.T","2531.T","8233.T","4502.T","6762.T","3401.T","4543.T",
    "8331.T","5631.T","9503.T","5101.T","9001.T","9602.T","5301.T",
    "8766.T","4043.T","9501.T","8035.T","9531.T","8804.T","9005.T",
    "3289.T","7911.T","3402.T","4042.T","5332.T","7203.T","8015.T",
    "4704.T","4208.T","9021.T","7951.T","7272.T","9064.T","6506.T",
    "6841.T"
]

# Download + clean using shared function
nikkei_clean = download_and_clean(nikkei_tickers)

# Save
nikkei_clean.to_csv("nikkei225_close_clean.csv")
print("Saved: nikkei225_close_clean.csv")

[*********************100%***********************]  225 of 225 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['9613.T']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (2953, 217)
Saved: nikkei225_close_clean.csv


# 🇦🇺 ASX 50 — Australia

In [9]:
asx50_tickers = [
    "CBA.AX","BHP.AX","CSL.AX","NAB.AX","WBC.AX","ANZ.AX","MQG.AX","WES.AX",
    "GMG.AX","WDS.AX","RIO.AX","TCL.AX","WOW.AX","ALL.AX","FMG.AX","STO.AX",
    "BXB.AX","COL.AX","TLS.AX","QBE.AX","ORG.AX","APA.AX","SCG.AX","SHL.AX",
    "S32.AX","IAG.AX","CPU.AX","JHX.AX","NST.AX","RMD.AX","MIN.AX","ALD.AX",
    "BSL.AX","ASX.AX","REA.AX","XRO.AX","WTC.AX","SEK.AX","CAR.AX","QAN.AX",
    "AZJ.AX","LYC.AX","IGO.AX","ILU.AX","ORI.AX","TWE.AX","EDV.AX","TLC.AX",
    "SVW.AX","NWS.AX"
]

# Download + clean using shared function
asx50_clean = download_and_clean(asx50_tickers)

# Save to CSV
asx50_clean.to_csv("asx50_close_clean.csv")
print("Saved: asx50_close_clean.csv")

[*********************100%***********************]  50 of 50 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SVW.AX']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (3037, 44)
Saved: asx50_close_clean.csv


#  🌍 MACRO DATA: FX + YIELDS + RISK FACTORS

In [10]:
def get_macro_data_with_indices(start="2014-01-01", end="2025-12-31"):
    """
    Enhanced version that includes global indices alongside macro factors.
    Uses ASX 50 as Australian market proxy.
    """
    # --------------------------------------------------------
    # 📉 1. FX RATES (Cross-currency structure)
    # --------------------------------------------------------
    fx_map = {
        'EURUSD': 'EURUSD=X',
        'GBPUSD': 'GBPUSD=X',
        'JPYUSD': 'JPYUSD=X',
        'AUDUSD': 'AUDUSD=X',
        'BRLUSD': 'BRLUSD=X',
        'ZARUSD': 'ZARUSD=X'
    }

    # --------------------------------------------------------
    # 📊 2. GLOBAL INDICES (Market proxies)
    # --------------------------------------------------------
    indices_map = {
        'SP500': '^GSPC',
        'FTSE100': '^FTSE',
        'EUROSTOXX50': '^STOXX50E',
        'Ibovespa': '^BVSP',
        'JSETop40': '^JTOPI',
        'NIKKEI225': '^N225',
        'ASX50': '^AORD'  # Using ASX All Ordinaries as proxy for ASX 50
    }

    # --------------------------------------------------------
    # 📈 3. RISK FACTORS (Global drivers)
    # --------------------------------------------------------
    risk_map = {
        'VIX': '^VIX',
        'Gold': 'GC=F',
        'Oil': 'CL=F',
        'Bitcoin': 'BTC-USD'
    }

    # Combine all Yahoo tickers
    yf_map = {**fx_map, **indices_map, **risk_map}
    yf_tickers = list(yf_map.values())

    print(f"Downloading {len(yf_tickers)} Yahoo series (FX, Indices, Risk Factors)...")

    yf_data = yf.download(yf_tickers, start=start, end=end)["Close"]
    yf_data = yf_data.rename(columns={v: k for k, v in yf_map.items()})

    # --------------------------------------------------------
    # 🏦 4. GOVERNMENT BOND YIELDS (FRED)
    # --------------------------------------------------------
    fred_map = {
        'US_10Y': 'DGS10',
        'DE_10Y': 'IRLTLT01DEM156N',
        'UK_10Y': 'IRLTLT01GBM156N',
        'JP_10Y': 'IRLTLT01JPM156N',
        'AU_10Y': 'IRLTLT01AUM156N',
        'BR_10Y': 'IRLTLT01BRM156N',
        'ZA_10Y': 'IRLTLT01ZAM156N'
    }

    print(f"Downloading {len(fred_map)} FRED series...")

    fred_data = pd.DataFrame()
    for name, ticker in fred_map.items():
        try:
            s = web.DataReader(ticker, "fred", start, end)
            fred_data[name] = s[ticker]
        except Exception as e:
            print(f"Failed: {name} ({ticker})")

    # --------------------------------------------------------
    # 🔗 MERGE ALL
    # --------------------------------------------------------
    macro = pd.concat([yf_data, fred_data], axis=1)

    # Cleaning
    macro = macro.dropna(axis=1, how='all')
    macro = macro.ffill().dropna()

    print("Final macro data shape:", macro.shape)
    print("\nIncluded series:")
    print("- FX Rates:", len(fx_map))
    print("- Global Indices:", len(indices_map))
    print("- Risk Factors:", len(risk_map))
    print("- Bond Yields:", len(fred_map))

    return macro

# RUN
macro_data_with_indices = get_macro_data_with_indices()

# SAVE
macro_data_with_indices.to_csv("macro_with_indices.csv")
print("\nSaved: macro_with_indices.csv")


/tmp/ipykernel_9174/748085689.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  yf_data = yf.download(yf_tickers, start=start, end=end)["Close"]
[                       0%                       ]

[*********************100%***********************]  17 of 17 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['^JTOPI']: YFPricesMissingError('possibly delisted; no price data found  (1d 2014-01-01 -> 2025-12-31)')


Failed: BR_10Y (IRLTLT01BRM156N)
Final macro data shape: (4124, 22)

Included series:
- FX Rates: 6
- Global Indices: 7
- Risk Factors: 4
- Bond Yields: 7

Saved: macro_with_indices.csv


In [11]:
def check_time_consistency(*dataframes, freq="B"):  # "B" = business days
    """Verify all DataFrames share the same time index and frequency."""
    if len(dataframes) < 1:
        raise ValueError("At least one DataFrame must be provided")

    # Start with the first index
    common_index = dataframes[0].index

    # Iteratively find intersection with all other indices
    for df in dataframes[1:]:
        common_index = common_index.intersection(df.index)

    # Check if all indices match the common index
    for df in dataframes:
        if not df.index.equals(common_index):
            raise ValueError("DataFrames have mismatched time indices!")

    # Check frequency (e.g., business days)
    for df in dataframes:
        inferred_freq = pd.infer_freq(df.index)
        if inferred_freq != freq:
            print(f"Warning: {df.columns[0] if len(df.columns) > 0 else 'DataFrame'} has frequency {inferred_freq}, expected {freq}")

    print(f"✅ All {len(dataframes)} DataFrames are time-consistent.")
    return common_index

def check_data_coverage(df, min_coverage=0.95):
    """Ensure no asset has >5% missing data."""
    coverage = df.notna().mean()
    low_coverage = coverage[coverage < min_coverage]
    if not low_coverage.empty:
        print(f"⚠️ Low coverage in: {low_coverage.index.tolist()}")
        print(f"Coverage values: {low_coverage.values}")
    return coverage

def convert_to_utc0(df):
    """Convert DataFrame index to UTC+0 (timezone-naive)."""
    df.index = pd.to_datetime(df.index).tz_localize(None)  # Remove timezone
    return df

def align_market_close(df, market_close_utc="16:00"):
    """Shift timestamps to a common UTC+0 market close time."""
    df.index = df.index.normalize() + pd.to_timedelta(market_close_utc)
    return df

# After downloading all datasets:
datasets = {
    "S&P 500": sp500_clean,
    "Ibovespa": bovespa_clean,
    "EURO STOXX 50": euro_clean,
    "FTSE 100": ftse_clean,
    "JSE Top 40": jse_clean,
    "NIKKEI 225": nikkei_clean,
    "ASX 50": asx50_clean,
    "Macro": macro_data_with_indices
}

# 1. Convert all to UTC+0
for name, df in datasets.items():
    datasets[name] = convert_to_utc0(df)

# 2. Check consistency
try:
    common_index = check_time_consistency(*datasets.values())
    print(f"Common index length: {len(common_index)}")

    # 3. Check coverage for each dataset
    for name, df in datasets.items():
        print(f"\n{name} coverage:")
        coverage = check_data_coverage(df)
        print(f"Mean coverage: {coverage.mean():.2%}")
except Exception as e:
    print(f"Error in consistency check: {str(e)}")
    # Continue with analysis even if there are inconsistencies
    # You might want to align the datasets to the common index
    # For example: aligned_df = df.reindex(common_index)


Error in consistency check: DataFrames have mismatched time indices!


In [12]:
from itertools import combinations
import pandas as pd

def analyze_time_mismatches(*datasets):
    """
    Analyzes and reports time index mismatches between multiple datasets.
    Returns the intersection of all dates (common index).
    """
    print("\n=== PAIRWISE MISMATCHES ===")

    # Create a dictionary to store all datasets with their names
    dataset_names = [f"Dataset {i+1}" for i in range(len(datasets))]
    named_datasets = {name: ds for name, ds in zip(dataset_names, datasets)}

    # Compare all pairs
    for (name1, ds1), (name2, ds2) in combinations(named_datasets.items(), 2):
        # Find mismatched dates
        mismatches = ds1.index.symmetric_difference(ds2.index)
        mismatch_count = len(mismatches)

        if mismatch_count > 0:
            print(f"{name1} vs {name2}: {mismatch_count} mismatched dates")
            print(f"  First 3 mismatches: {list(mismatches)[:3]}")
        else:
            print(f"{name1} vs {name2}: Perfect match")

    # Find common index (intersection of all dates)
    common_index = datasets[0].index
    for ds in datasets[1:]:
        common_index = common_index.intersection(ds.index)

    print(f"\nCommon dates across all datasets: {len(common_index)}")
    return common_index


In [14]:
# After downloading all datasets:
datasets = {
    "S&P 500": sp500_clean,
    "Ibovespa": bovespa_clean,
    "EURO STOXX 50": euro_clean,
    "FTSE 100": ftse_clean,
    "JSE Top 40": jse_clean,
    "NIKKEI 225": nikkei_clean,
    "ASX 50": asx50_clean,
    "Macro": macro_data_with_indices
}

# Convert all to UTC+0 first
for name, df in datasets.items():
    datasets[name] = convert_to_utc0(df)

# Analyze mismatches
common_index = analyze_time_mismatches(*datasets.values())



=== PAIRWISE MISMATCHES ===
Dataset 1 vs Dataset 2: 302 mismatched dates
  First 3 mismatches: [Timestamp('2014-01-23 00:00:00'), Timestamp('2014-01-24 00:00:00'), Timestamp('2014-01-27 00:00:00')]
Dataset 1 vs Dataset 3: 120 mismatched dates
  First 3 mismatches: [Timestamp('2014-07-10 00:00:00'), Timestamp('2014-07-11 00:00:00'), Timestamp('2014-07-14 00:00:00')]
Dataset 1 vs Dataset 4: 263 mismatched dates
  First 3 mismatches: [Timestamp('2014-01-02 00:00:00'), Timestamp('2014-01-03 00:00:00'), Timestamp('2014-01-06 00:00:00')]
Dataset 1 vs Dataset 5: 305 mismatched dates
  First 3 mismatches: [Timestamp('2014-01-01 00:00:00'), Timestamp('2014-01-02 00:00:00'), Timestamp('2014-01-03 00:00:00')]
Dataset 1 vs Dataset 6: 395 mismatched dates
  First 3 mismatches: [Timestamp('2014-01-06 00:00:00'), Timestamp('2014-01-07 00:00:00'), Timestamp('2014-01-08 00:00:00')]
Dataset 1 vs Dataset 7: 275 mismatched dates
  First 3 mismatches: [Timestamp('2014-01-02 00:00:00'), Timestamp('2014-01-

In [15]:
def convert_to_utc0(df):
    """
    Convert DataFrame index to UTC+0 timezone.
    If already timezone-naive, assumes it's UTC.
    """
    if df.index.tz is not None:
        return df.tz_convert('UTC')
    return df


In [16]:
def streamline_datasets(datasets, start_date='2015-01-01', end_date='2025-12-30'):
    """
    Align all datasets to a common date range and index.
    Returns aligned datasets and common index.
    """
    print("\n=== STREAMLINING DATASETS ===")

    # Convert string dates to datetime
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # First, align all datasets to the same date range
    aligned_datasets = {}
    for name, df in datasets.items():
        # Slice to the desired date range
        df = df.loc[start_date:end_date]

        # Store the aligned dataset
        aligned_datasets[name] = df
        print(f"{name}: {len(df)} dates ({df.index.min()} to {df.index.max()})")

    # Now find the common index across all aligned datasets
    common_index = None
    for df in aligned_datasets.values():
        if common_index is None:
            common_index = df.index
        else:
            common_index = common_index.intersection(df.index)

    print(f"\nCommon dates after alignment: {len(common_index)} ({common_index.min()} to {common_index.max()})")

    # Reindex all datasets to the common index
    final_datasets = {}
    for name, df in aligned_datasets.items():
        final_datasets[name] = df.reindex(common_index)
        print(f"{name} final: {len(final_datasets[name])} dates")

    return final_datasets, common_index

# Apply the streamlining
streamlined_datasets, common_index = streamline_datasets(datasets)

# Verify the alignment
print("\n=== VERIFICATION ===")
for name, df in streamlined_datasets.items():
    print(f"{name}: {df.index.min()} to {df.index.max()} ({len(df)} dates)")

# Check if all datasets now have the same index
all_indices_match = all(df.index.equals(common_index) for df in streamlined_datasets.values())
print(f"\nAll datasets have identical indices: {all_indices_match}")

# Save the streamlined datasets if needed
for name, df in streamlined_datasets.items():
    df.to_csv(f"streamlined_{name.replace(' ', '_').lower()}.csv")
    print(f"Saved: streamlined_{name.replace(' ', '_').lower()}.csv")



=== STREAMLINING DATASETS ===
S&P 500: 2765 dates (2015-01-02 00:00:00 to 2025-12-30 00:00:00)
Ibovespa: 2737 dates (2015-01-02 00:00:00 to 2025-12-30 00:00:00)
EURO STOXX 50: 2817 dates (2015-01-02 00:00:00 to 2025-12-30 00:00:00)
FTSE 100: 2778 dates (2015-01-02 00:00:00 to 2025-12-30 00:00:00)
JSE Top 40: 2776 dates (2015-01-01 00:00:00 to 2025-12-30 00:00:00)
NIKKEI 225: 2709 dates (2015-01-05 00:00:00 to 2025-12-30 00:00:00)
ASX 50: 2784 dates (2015-01-02 00:00:00 to 2025-12-30 00:00:00)
Macro: 4017 dates (2015-01-01 00:00:00 to 2025-12-30 00:00:00)

Common dates after alignment: 2413 (2015-01-05 00:00:00 to 2025-12-30 00:00:00)
S&P 500 final: 2413 dates
Ibovespa final: 2413 dates
EURO STOXX 50 final: 2413 dates
FTSE 100 final: 2413 dates
JSE Top 40 final: 2413 dates
NIKKEI 225 final: 2413 dates
ASX 50 final: 2413 dates
Macro final: 2413 dates

=== VERIFICATION ===
S&P 500: 2015-01-05 00:00:00 to 2025-12-30 00:00:00 (2413 dates)
Ibovespa: 2015-01-05 00:00:00 to 2025-12-30 00:00:0

In [17]:
def extract_first_period(streamlined_datasets, end_date='2021-12-31'):
    """
    Extract first period from streamlined datasets up to specified end date.
    Returns dictionary of datasets for first period.
    """
    end_date = pd.to_datetime(end_date)

    print("\n=== EXTRACTING FIRST PERIOD ===")

    # Find the actual end date to use
    actual_end = None
    for df in streamlined_datasets.values():
        # Get dates on or before desired end date
        available_dates = df.index[df.index <= end_date]
        if len(available_dates) > 0:
            candidate = available_dates.max()
            if actual_end is None or candidate < actual_end:
                actual_end = candidate

    if actual_end is None:
        raise ValueError("No dates available before the specified end date")

    print(f"Using end date: {actual_end.date()}")

    # Extract first period for each dataset
    first_period = {}
    for name, df in streamlined_datasets.items():
        first_period[name] = df.loc[:actual_end]
        print(f"{name}: {len(first_period[name])} dates ({df.index.min().date()} to {actual_end.date()})")

    return first_period, actual_end

# Extract the first period
first_period_datasets, period_end = extract_first_period(streamlined_datasets, '2021-12-31')

# Save the first period datasets
print("\n=== SAVING FIRST PERIOD DATASETS ===")
for name, df in first_period_datasets.items():
    filename = f"first_period_{name.replace(' ', '_').lower()}.csv"
    df.to_csv(filename)
    print(f"Saved: {filename}")

print(f"\nFirst period extracted through {period_end.date()}")



=== EXTRACTING FIRST PERIOD ===
Using end date: 2021-12-30
S&P 500: 1548 dates (2015-01-05 to 2021-12-30)
Ibovespa: 1548 dates (2015-01-05 to 2021-12-30)
EURO STOXX 50: 1548 dates (2015-01-05 to 2021-12-30)
FTSE 100: 1548 dates (2015-01-05 to 2021-12-30)
JSE Top 40: 1548 dates (2015-01-05 to 2021-12-30)
NIKKEI 225: 1548 dates (2015-01-05 to 2021-12-30)
ASX 50: 1548 dates (2015-01-05 to 2021-12-30)
Macro: 1548 dates (2015-01-05 to 2021-12-30)

=== SAVING FIRST PERIOD DATASETS ===
Saved: first_period_s&p_500.csv
Saved: first_period_ibovespa.csv
Saved: first_period_euro_stoxx_50.csv
Saved: first_period_ftse_100.csv
Saved: first_period_jse_top_40.csv
Saved: first_period_nikkei_225.csv
Saved: first_period_asx_50.csv
Saved: first_period_macro.csv

First period extracted through 2021-12-30
